In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# ==============================================================================
# CONFIGURAZIONE
# ==============================================================================
DIR_CSV = "risultati_csv" # Assicurati che il nome della cartella sia corretto

# Periodi temporali (Impostati su tutto il periodo a disposizione)
ANNI_STORICI = (1950, 2015) # Passato: 1950 - 2015
ANNI_FUTURI = (2016, 2100)  # Futuro: 2016 - 2100

# Variabili da analizzare
VARIABILI = {
    'Frequenza': 'Num_Ondate',
    'Durata': 'Durata_Media_Giorni',
    'Intensita': 'Intensita_Media_K'
}

# ==============================================================================
# 1. CARICAMENTO DATI E CALCOLO DELTA PER SINGOLO MODELLO
# ==============================================================================
def calcola_delta_modelli():
    file_csv = glob.glob(os.path.join(DIR_CSV, "*.csv"))
    if not file_csv:
        print(f"ERRORE: Nessun file CSV trovato in '{DIR_CSV}'.")
        return None

    risultati_delta = []
    
    for file in file_csv:
        modello = os.path.basename(file).replace("Risultati_HWI_Aprile_Ottobre_", "").replace(".csv", "")
        df = pd.read_csv(file)
        
        for icao in df['Aeroporto'].unique():
            df_aero = df[df['Aeroporto'] == icao]
            
            # Applichiamo i filtri esatti (1950-2015) e (2016-2100)
            df_hist = df_aero[(df_aero['Scenario'] == 'historical') & (df_aero['Anno'] >= ANNI_STORICI[0]) & (df_aero['Anno'] <= ANNI_STORICI[1])]
            df_fut = df_aero[(df_aero['Scenario'] == 'ssp585') & (df_aero['Anno'] >= ANNI_FUTURI[0]) & (df_aero['Anno'] <= ANNI_FUTURI[1])]
            
            # Calcolo le medie di Frequenza, Durata e Intensità per i due periodi globali
            medie_hist = df_hist[list(VARIABILI.values())].mean()
            medie_fut = df_fut[list(VARIABILI.values())].mean()
            
            # Calcolo i Delta
            delta = medie_fut - medie_hist
            
            row = {'Aeroporto': icao, 'Modello': modello}
            for nome_var, col_var in VARIABILI.items():
                row[f'Delta_{nome_var}'] = delta[col_var]
                
            risultati_delta.append(row)
            
    return pd.DataFrame(risultati_delta)

# ==============================================================================
# 2. STATISTICA DI ENSEMBLE E INCERTEZZE
# ==============================================================================
def calcola_statistiche_ensemble(df_delta):
    df_stats = pd.DataFrame()
    aeroporti = df_delta['Aeroporto'].unique()
    
    for icao in aeroporti:
        df_aero = df_delta[df_delta['Aeroporto'] == icao]
        row_stat = {'Aeroporto': icao}
        
        for var in VARIABILI.keys():
            col_name = f'Delta_{var}'
            valori = df_aero[col_name].dropna()
            
            if len(valori) == 0:
                continue
                
            mean_val = np.mean(valori)
            std_val = np.std(valori, ddof=1)
            p10 = np.percentile(valori, 10)
            p90 = np.percentile(valori, 90)
            concordanza = (np.sum(valori > 0) / len(valori)) * 100
            snr = mean_val / std_val if std_val != 0 else np.nan
            
            row_stat[f'{var}_Media'] = round(mean_val, 2)
            row_stat[f'{var}_StdDev'] = round(std_val, 2)
            row_stat[f'{var}_10p'] = round(p10, 2)
            row_stat[f'{var}_90p'] = round(p90, 2)
            row_stat[f'{var}_Concordanza_%'] = round(concordanza, 1)
            row_stat[f'{var}_SNR'] = round(snr, 2)
            
        df_stats = pd.concat([df_stats, pd.DataFrame([row_stat])], ignore_index=True)
        
    return df_stats

# ==============================================================================
# ESECUZIONE MAIN E SALVATAGGIO
# ==============================================================================
df_delta_tutti = calcola_delta_modelli()
if df_delta_tutti is not None:
    df_statistiche = calcola_statistiche_ensemble(df_delta_tutti)
    
    # Salva il file Excel DEFINITIVO
    df_statistiche.to_excel('Risultati_Punto1_Incertezza.xlsx', index=False)
    print("✓ Tabella Statistica Excel aggiornata a 1950-2100 e salvata con successo!")

# Mostriamo l'anteprima in Jupyter Lab
df_statistiche.head()




✓ Tabella Statistica Excel aggiornata a 1950-2100 e salvata con successo!


,Aeroporto,Frequenza_Media,Frequenza_StdDev,Frequenza_10p,Frequenza_90p,Frequenza_Concordanza_%,Frequenza_SNR,Durata_Media,Durata_StdDev,Durata_10p,Durata_90p,Durata_Concordanza_%,Durata_SNR,Intensita_Media,Intensita_StdDev,Intensita_10p,Intensita_90p,Intensita_Concordanza_%,Intensita_SNR
0,EBBR,2.24,1.26,1.09,3.40,100.0,1.78,1.74,0.99,0.78,2.54,96.8,1.76,0.57,0.44,0.18,1.10,96.8,1.29
1,EDDF,2.39,1.31,1.20,3.40,100.0,1.82,1.85,0.99,0.69,2.74,96.8,1.87,0.61,0.47,0.19,0.96,93.5,1.30
2,EDDL,2.16,1.32,1.07,3.21,100.0,1.64,1.53,1.01,0.30,2.48,96.8,1.52,0.47,0.42,0.02,0.93,90.3,1.13
3,EDDM,2.83,1.17,1.77,3.68,100.0,2.42,2.98,1.63,1.36,4.32,100.0,1.83,0.55,0.52,0.08,1.09,93.5,1.08
4,EGKK,2.31,1.00,0.98,3.09,100.0,2.32,3.20,2.18,1.11,6.14,100.0,1.47,0.44,0.28,0.13,0.79,96.8,1.56


In [2]:
import pandas as pd
df = pd.read_excel('Risultati_Punto1_Incertezza.xlsx')

print("--- TOP 5 AEROPORTI PER AUMENTO FREQUENZA ONDATE ---")
print(df.sort_values(by='Frequenza_Media', ascending=False)[['Aeroporto', 'Frequenza_Media', 'Frequenza_Concordanza_%']].head())

print("\n--- TOP 5 AEROPORTI PER AUMENTO DURATA ---")
print(df.sort_values(by='Durata_Media', ascending=False)[['Aeroporto', 'Durata_Media', 'Durata_Concordanza_%']].head())

print("\n--- STATISTICHE MEDIE GLOBALI ---")
print(f"Concordanza media Frequenza: {df['Frequenza_Concordanza_%'].mean():.1f}%")
print(f"Concordanza media Durata: {df['Durata_Concordanza_%'].mean():.1f}%")
print(f"SNR Medio Frequenza: {df['Frequenza_SNR'].mean():.2f}")

--- TOP 5 AEROPORTI PER AUMENTO FREQUENZA ONDATE ---
   Aeroporto  Frequenza_Media  Frequenza_Concordanza_%
58      MMUN             4.48                    100.0
15      LESO             3.35                    100.0
34      KLAX             3.33                    100.0
37      KMIA             3.00                    100.0
56      CYHZ             2.98                    100.0

--- TOP 5 AEROPORTI PER AUMENTO DURATA ---
   Aeroporto  Durata_Media  Durata_Concordanza_%
14      LEPA         17.54                 100.0
20      LICG         16.96                 100.0
12      LEBL         12.97                 100.0
58      MMUN         11.49                 100.0
19      LGHI         10.83                 100.0

--- STATISTICHE MEDIE GLOBALI ---
Concordanza media Frequenza: 99.6%
Concordanza media Durata: 98.4%
SNR Medio Frequenza: 2.41
